# task_14 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

policies = pd.read_csv(base / "policies.csv")
claims = pd.read_csv(base / "claims.csv")
payments = pd.read_csv(base / "payments.csv")

claims["loss_date"] = pd.to_datetime(claims["loss_date"])
claims["report_date"] = pd.to_datetime(claims["report_date"])
claims["close_date"] = pd.to_datetime(claims["close_date"])

merged = claims.merge(policies[["policy_id", "product"]], on="policy_id")
q1 = str(merged.groupby("product")["claim_amount"].mean().sort_values(ascending=False).index[0])

paid_total = payments.groupby("claim_id")["amount_paid"].sum().rename("paid_total")
claims_paid = claims.join(paid_total, on="claim_id")
collision = claims_paid[claims_paid["cause"] == "Collision"]
q2 = f"{(collision['paid_total'] / collision['claim_amount']).median():.3f}"

count = 0
for _, group in claims.sort_values("loss_date").groupby("policy_id"):
    if (group["loss_date"].diff().dt.days.dropna() <= 30).any():
        count += 1
q3 = str(count)

claims["settlement_days"] = (claims["close_date"] - claims["report_date"]).dt.days
q4 = str(claims.groupby("adjuster")["settlement_days"].mean().sort_values(ascending=False).index[0])

claims_paid["loss_month"] = claims_paid["loss_date"].dt.to_period("M").astype(str)
q5 = str(claims_paid.groupby("loss_month")["paid_total"].sum().sort_values(ascending=False).index[0])


Q1: Which product line has the highest average claim_amount?

In [ ]:
print(q1)


Q2: For claims caused by Collision, what is the median payout ratio paid_total / claim_amount, rounded to 3 decimals?

In [ ]:
print(q2)


Q3: How many policyholders had multiple claims with loss_date values within 30 days of each other?

In [ ]:
print(q3)


Q4: Which adjuster has the highest average settlement_days, where settlement_days = close_date - report_date?

In [ ]:
print(q4)


Q5: Using loss_date month, which month has the highest total paid_total?

In [ ]:
print(q5)
